# SQL Practice - Day 2

Each question includes DataFrames (`df`, `df1` where needed).

## Q1

**Task:** Find the customer(s) with the **2nd highest total spending**.

In [1]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6,7],'customer_id':[101,101,102,103,102,101,104],'amount':[500,700,300,1000,200,800,1500]})
df

,order_id,customer_id,amount
0,1,101,500
1,2,101,700
2,3,102,300
3,4,103,1000
4,5,102,200
5,6,101,800
6,7,104,1500


In [17]:
from pandasql import sqldf
sqldf("""
        with my_cte as (
        select customer_id , sum(amount) as total_amount
        from df
        group by customer_id
        )
        select customer_id , total_amount
        from (select customer_id , total_amount,
                dense_rank()
                over(order by total_amount desc) as rn
                from my_cte)
        where rn = 2
                



""")

,customer_id,total_amount
0,104,1500


## Q2

**Task:** Find employees whose **salary is greater than the average salary of their department**.

In [18]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'name':['A','B','C','D','E','F'],'department':['HR','HR','HR','IT','IT','IT'],'salary':[50000,60000,70000,40000,45000,80000]})
df

,emp_id,name,department,salary
0,1,A,HR,50000
1,2,B,HR,60000
2,3,C,HR,70000
3,4,D,IT,40000
4,5,E,IT,45000
5,6,F,IT,80000


In [19]:
sqldf("""
        select name , salary
        from df a
        where salary > (select avg(salary)
                            from df b
                            where a.department = b.department)



""")

,name,salary
0,C,70000
1,F,80000


## Q3

**Task:** Find the **highest-priced product in each category**.

In [20]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,150,200,250,300,350]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,250
4,5,B,300
5,6,B,350


In [22]:
sqldf("""
        select product_id , category ,price
        from (select product_id , category ,price,
            dense_rank()
            over(partition by category order by price desc) as rn
            from df)
        where rn = 1



""")

,product_id,category,price
0,3,A,200
1,6,B,350


## Q4

**Task:** Return **customer name and total spending** (include customers with no orders).

In [23]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'customer_name':['Alice','Bob','Charlie','David']})
df1=pd.DataFrame({'order_id':[1,2,3,4],'customer_id':[101,101,102,103],'amount':[500,700,300,1000]})
df,df1

(   customer_id customer_name
 0          101         Alice
 1          102           Bob
 2          103       Charlie
 3          104         David,
    order_id  customer_id  amount
 0         1          101     500
 1         2          101     700
 2         3          102     300
 3         4          103    1000)

In [27]:
sqldf("""
        with my_cte as (
        select e.customer_id as customer_Id , e.customer_name as customer_name ,o.order_id as order_id , o.amount as amount
        from df e
        left join df1 o on e.customer_id = o.customer_id
        )
        select customer_name , sum(amount) 
        from my_cte
        group by customer_name
        having sum(amount) = (select max(sum_amount) 
                                from (select customer_name , sum(amount) as sum_amount
                                        from my_cte
                                        group by customer_name))



""")

,customer_name,sum(amount)
0,Alice,1200


## Q5

**Task:** Find the **top 3 highest salaries in the company**.

In [28]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'name':['A','B','C','D','E','F'],'salary':[50000,60000,70000,40000,45000,80000]})
df

,emp_id,name,salary
0,1,A,50000
1,2,B,60000
2,3,C,70000
3,4,D,40000
4,5,E,45000
5,6,F,80000


In [43]:
sqldf("""
        select name ,salary
        from (select name , salary,
                row_number()
                over(order by salary desc) as rn 
                from df ) 
        where rn <= 3 -- avoids duplicate pairs and self-pairs



""")

,employee1,employee2,manager_name,salary
0,B,C,A,60000
1,D,E,B,70000


## Q6

**Task:** Find employee pairs with the **same manager and same salary**.

In [34]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'name':['A','B','C','D','E','F'],'manager_id':[None,1,1,2,2,2],'salary':[100000,60000,60000,70000,70000,90000]})
df

,emp_id,name,manager_id,salary
0,1,A,NaN,100000
1,2,B,1.0,60000
2,3,C,1.0,60000
3,4,D,2.0,70000
4,5,E,2.0,70000
5,6,F,2.0,90000


In [44]:
sqldf("""
        SELECT
    e1.name AS employee1,
    e2.name AS employee2,
    m.name AS manager_name,
    e1.salary
FROM df e1
JOIN df e2
    ON e1.manager_id = e2.manager_id
   AND e1.salary = e2.salary
   AND e1.emp_id < e2.emp_id     
JOIN df m
    ON e1.manager_id = m.emp_id;
    
        



""")

,employee1,employee2,manager_name,salary
0,B,C,A,60000
1,D,E,B,70000


## Q7

**Task:** Using a CTE, find departments whose **average salary is greater than the company average salary**.

In [45]:
import pandas as pd
df=pd.DataFrame({'department':['HR','HR','HR','IT','IT','IT','Sales','Sales'],'salary':[50000,60000,70000,40000,45000,80000,90000,85000]})
df

,department,salary
0,HR,50000
1,HR,60000
2,HR,70000
3,IT,40000
4,IT,45000
5,IT,80000
6,Sales,90000
7,Sales,85000


In [47]:
sqldf("""
        with my_cte as (
        select department, avg(salary) as avg_salary
        from df
        group by department
        having avg(salary) > (select avg(salary) 
                                from df)
        )
        select * from my_cte




""")

,department,avg_salary
0,Sales,87500.0


## Q8

**Task:** Find products whose **price is between the category average and category maximum**.

In [49]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6],'category':['A','A','A','B','B','B'],'price':[100,150,200,250,300,350]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,250
4,5,B,300
5,6,B,350


In [52]:
sqldf("""
        select product_id , category , price
        from df a
        where price >= (select avg(price) 
                        from df b
                        where a.category = b.category )
        and price < (select max(price) 
                        from df c
                        where a.category = c.category)



""")

,product_id,category,price
0,2,A,150
1,5,B,300


## Q9

**Task:** Find each customer's **first order date and latest order date**.

In [59]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5,6],'customer_id':[101,101,102,103,102,101],'order_date':['2025-01-01','2025-01-15','2025-01-05','2025-01-10','2025-01-20','2025-02-01']})
df

,order_id,customer_id,order_date
0,1,101,2025-01-01
1,2,101,2025-01-15
2,3,102,2025-01-05
3,4,103,2025-01-10
4,5,102,2025-01-20
5,6,101,2025-02-01


In [60]:
sqldf("""
        select customer_id , max(order_date) as latest_order , min(order_date) as first_order
        from df
        group by customer_id



""")

,customer_id,latest_order,first_order
0,101,2025-02-01,2025-01-01
1,102,2025-01-20,2025-01-05
2,103,2025-01-10,2025-01-10


## Q10

**Task:** Find the **3rd highest priced product in each category**.

In [55]:
import pandas as pd
df=pd.DataFrame({'product_id':[1,2,3,4,5,6,7,8],'category':['A','A','A','B','B','B','C','C'],'price':[100,150,200,250,300,350,500,450]})
df

,product_id,category,price
0,1,A,100
1,2,A,150
2,3,A,200
3,4,B,250
4,5,B,300
5,6,B,350
6,7,C,500
7,8,C,450


In [58]:
sqldf("""
select product_id , category , price
from (select product_id , category , price,
        Dense_rank()
        over(partition by category order by price desc) as rn
        from df)
where rn = 3 
""")

,product_id,category,price
0,1,A,100
1,4,B,250
